# Milestone 6 Lab, notebook 6 of 8: The end-to-end evaluation

## Where we are

| Notebook | What you do |
|---|---|
| 1. Late and corrected files, by hand | Work out by hand what three late files should do to the figures |
| 2. Corrections and incremental runs | Load the late files, recalculate only what they touch, and prove it equals a full rebuild |
| 3. The review desk | Decide the findings that need review, and watch the next run obey |
| 4. Run health and restatements | Read a run's health, and ask the analyst why a reported figure changed |
| 5. A judge with a reference answer | Test a judge on explanations whose grade you know, then use it |
| **6. The end-to-end evaluation** (you are here) | Score the whole system from files to answers, on two question sets |
| 7. The improver | An agent proposes a change to the analyst, you approve, and code keeps it or rolls it back |
| 8. The whole system | Run everything from files to answer, and look back over six milestones |

Model requests in this notebook: **about 70**. A run of this lab with its default small sets costs about 10 US cents.

## What this notebook is for

Every evaluation so far started from data that was already built. This one starts from **nothing**: it writes the files, loads them, runs the pipeline, loads the late files, runs again, and only then asks. If any layer of the stack is broken, the score shows it.

That is an **end-to-end evaluation**. You also meet two ideas you need before letting anything improve the analyst:

- a **held-out set** of questions, kept apart;
- **noise**: how much a score moves by chance.

> This notebook needs `OPENROUTER_API_KEY`. With the default small sets it makes about 30 requests and costs about 3 US cents.

In [ ]:
import time

import pandas as pd
from lab_helpers import LAB_E2E, show_doc

## Step 1: The design

In [ ]:
show_doc("HLD.md", "8. The end-to-end evaluation")

## Step 2: Build everything from nothing

In [ ]:
from retail_profit_stack.e2e import build_everything, open_stores

start = time.time()
workspace = build_everything(LAB_E2E)
stores = open_stores(workspace)

print("Built in", round(time.time() - start, 1), "seconds")
print("Files:", len(list(workspace.places.landing.glob("*.csv"))), "on time,", len(list(workspace.late.glob("*.csv"))), "late")
print("Runs: ", sorted(folder.name for folder in (workspace.warehouse / "runs").iterdir()))

About a second, with no model. Six milestones of work: generating, loading, checking, quarantining, calculating, tracing, recording.

## Step 3: Two sets of questions

The **evaluation set** is the one we have been building since milestone 1. It now has 18 questions. Four of them only make sense after the late files, so each question has a **stage**.

In [ ]:
from retail_profit_stack.evaluation import load_set
from retail_profit_stack.scoring import right_level

pd.set_option("display.max_colwidth", 95)


def describe(cases):
    return pd.DataFrame([{
        "case_id": case.case_id, "stage": case.stage, "tiny": case.tiny,
        "judged": case.reference is not None, "right_level": right_level(case), "question": case.question,
    } for case in cases])


describe(load_set("evaluation"))

`judged` marks the questions with a reference answer, which the judge grades.

The **held-out set** is new: ten questions of the same kinds, about other products and months.

In [ ]:
describe(load_set("held-out"))

Why a second set? In the next notebook, an AI improver reads the analyst's mistakes on the evaluation set and writes rules to fix them. A rule written for one question fixes that question, and the evaluation score goes up, whether or not the rule helps anywhere else. That is called **overfitting**.

The held-out set is the defence. Nobody improving the analyst ever reads it. If a change is real, the held-out score moves too.

## Step 4: How many questions

`TINY = True` keeps the questions marked tiny: 6 and 5. Set it to `False` for all 18 and 10, which costs about 6 US cents for this notebook.

In [ ]:
TINY = True

evaluation_cases = load_set("evaluation", tiny=TINY)
held_out_cases = load_set("held-out", tiny=TINY)

print(len(evaluation_cases), "evaluation questions:", [case.case_id for case in evaluation_cases])
print(len(held_out_cases), "held-out questions:  ", [case.case_id for case in held_out_cases])

## Step 5: One question, by hand

`answer_and_score` is the evaluation for one question. It picks the store for the question's stage, asks the analyst, scores the answer with milestone 4's scoring, and has the judge grade the reason if the question has a reference.

In [ ]:
from retail_profit_stack.analyst_versions import version_one
from retail_profit_stack.e2e import answer_and_score
from retail_profit_stack.judge import make_judge

judge = make_judge()
analyst = version_one()

late_case = next(case for case in evaluation_cases if case.stage == "late")
print(late_case.case_id, late_case.stage, "-", late_case.question)

row = answer_and_score(late_case, analyst, stores, judge)
row

`version_one()` is the analyst as milestone 5 left it: its instructions and its model. From here on the analyst has **versions**, because the next notebook makes new ones.

## Step 6: The evaluation set

`evaluate` does that for every question, several at once, using milestone 5's dispatcher.

In [ ]:
from retail_profit_stack.e2e import evaluate, summary

scoreboard = []


def measure(cases, set_name):
    """Run the evaluation on one set, show the table, and add its scores to the scoreboard."""
    table = evaluate(cases, workspace, config=analyst, judge=judge, at_once=6)
    scoreboard.append({"set": set_name, **summary(table)})
    return table


evaluation_table = measure(evaluation_cases, "evaluation")

evaluation_table[["case_id", "stage", "right_level", "level", "right", "mistake", "letter"]]

In [ ]:
summary(evaluation_table)

How to read the scores:

- **accuracy:** the share of questions where every check passed.
- **grade:** the judge's average grade on the judged questions, from 0 (F) to 3 (A).
- **cost_of_mistakes:** from the business's cost table in milestone 4.
- **cost_per_question** and **seconds_per_question:** what the analyst itself cost.

Any wrong answers, with the checks that failed:

In [ ]:
evaluation_table[~evaluation_table["right"]][["case_id", "mistake", "failed_checks", "headline"]]

An empty table means every question passed this time.

## Step 7: The held-out set, twice

Now the held-out questions. We run them **twice**, with nothing changed in between.

In [ ]:
first = measure(held_out_cases, "held-out")
second = measure(held_out_cases, "held-out")

pd.DataFrame(scoreboard)

Compare the two held-out rows. Nothing changed between them: the same analyst, the same questions, the same data. Yet the scores may differ.

That difference is **noise**. Look hardest at `seconds_per_question`. When this lab was built, the same analyst took 4.8 seconds a question in one run and 10.8 in the next, because one model call stalled. Accuracy moved by a question.

This matters for one reason. Suppose we change the analyst and the score goes up by one question. Did the change help, or was it chance? **A change must beat the noise before we believe it.** With five questions, one question is twenty points, so the small set cannot tell. Even the full set is rough.

## Step 8: What the held-out set already found

When this lab was built, the first held-out run found something the evaluation set never had. The question "What will net sales be in December 2026?" was correctly declined, and the verifier **failed** the answer: it read "2026" in "December 2026" as a number no tool had returned.

That is a bug in the verifier, not in the analyst. It had been there since milestone 1. The evaluation set's forecast question says "next quarter", with no year, so it never triggered it. Ten new questions, written only to be different, found it on their first run. It is fixed now, and there is a test for it.

A second set of questions is worth having even before anything is improved.

## What we now know

- An **end-to-end evaluation** starts from nothing and builds every layer before it asks. Ours builds in about a second.
- Questions have a **stage**: some are asked at close, some after the late files.
- A **held-out set** is kept away from whoever improves the system, to catch **overfitting**.
- Scores have **noise**. The same analyst scored differently on two runs. Time moved most.
- A change only counts if it beats the noise.

**Next:** notebook 7. An agent proposes changes to the analyst, you approve them, and code uses these scores to keep or roll back.